# 01 — Data exploration: raw-material price forecasting

**Goal of the challenge:** forecast raw-material prices **3, 6 and 9 months ahead** for the materials in two
Schneider Electric Ring Main Units (RM AirSeT and RM6 NE-IQI), then use them in a business recommendation.

**Goal of this notebook:** take stock of the data before modelling.

1. What files do we have, and what is in each one?
2. What is missing (gaps, empty series, publication lags, unreadable files)?
3. Which series are **targets** (what we forecast), and which are **features** (what we forecast with)?
4. How much training data does each horizon leave us?

The data comes as `data/Data.zip`, with two more zips nested inside. The first cell extracts everything into `data/Data/`
if it is not there yet. That folder is gitignored.

**Findings** are summarised at the end (section 11).

In [ ]:
from pathlib import Path
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from cycler import cycler

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA = ROOT / "data"
RAW = DATA / "Data"                       # content of Data.zip
CSV = RAW / "data"                        # content of the nested data.zip (official challenge dataset)
SUP = RAW / "Supplier_balancesheet"       # content of the nested Supplier_balancesheet.zip


def extract(zip_path: Path, dest: Path) -> None:
    """Unzip, skipping macOS resource-fork junk."""
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest, members=[m for m in z.namelist() if not m.startswith("__MACOSX")])


if not RAW.exists():
    extract(DATA / "Data.zip", DATA)
if not CSV.exists():
    extract(RAW / "data.zip", RAW)
if not SUP.exists():
    extract(RAW / "Supplier_balancesheet.zip", RAW)

# Chart style: categorical colours always assigned in this fixed order.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
GREY = "#8a8984"
plt.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "#fcfcfb",
    "axes.facecolor": "#fcfcfb",
    "axes.prop_cycle": cycler(color=PALETTE),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#b5b4ae",
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e6e5e0",
    "grid.linewidth": 0.6,
    "axes.titlesize": 10,
    "axes.titleweight": "bold",
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "legend.frameon": False,
    "lines.linewidth": 1.6,
})

ASOF = pd.Timestamp("2026-11-12")   # download date stated in the data README
print("Repo root:", ROOT)

## 1. File inventory

Everything shipped in `Data.zip`, after extracting the nested zips.

In [ ]:
files = sorted(p for p in RAW.rglob("*") if p.is_file() and p.suffix != ".zip")
inv = pd.DataFrame({
    "file": [str(p.relative_to(RAW)) for p in files],
    "kb": [round(p.stat().st_size / 1024, 1) for p in files],
})
inv

What each source is, and what role it can play. *Target* = something we forecast. *Feature* = an input to the
forecast. *Context* = useful for the business recommendation but not as a monthly model input.

| Source | Content | Frequency / period | Role |
|---|---|---|---|
| `data/rm_prices_monthly_wide.csv` (+ `_long`, `source_catalog`, `data_dictionary`, xlsx) | 13 public price series and price indices (LME metals, Eurostat & BLS PPIs, EUR/USD), 2 derived EUR prices, 8 **empty** licensed columns | monthly, 2016-11 → 2026-09 | **targets** (Cu, Al, plastics) + **features** |
| `data/energy_prices_monthly_wide.csv` (+ `_long`, catalog, xlsx) | 30 energy series: oil, gas, power, coal, energy PPIs, Russian crude share, FX | monthly, 2016-11 → 2026-10 | **features** (leading indicators) |
| `data/bom_materials.csv` | Material mass / share for the 2 products (Schneider internal) | static | weights for targets |
| `RM_Prices_XHEC_Challenge.xlsx` (top level) | A **second, different** version of the BOM | static | to reconcile with the CSV BOM |
| `Copia de Europe Steel INDEXES.xlsx` | **Licensed** steel prices: MEPS & CRU flat (HR/CR/HDG coil), CRU stainless (304/430/316), CRU long products, Fastmarkets Turkey. Also a Schneider part → index mapping sheet | monthly (CRU flat = weekly + average), 1997/2008/2012 → 2026-10 | **targets** (carbon, galvanised, stainless steel) |
| `LME_Copper_Aluminium_EU_Monthly_2016_2026.xlsx` | LME Cu & Al cash **and 3-month** prices, EUR conversion, **LME stocks**, Nov-2026 month-to-date | monthly, 2016-01 → 2026-11 (MTD) | extends Cu/Al targets + **features** (spread, stocks) |
| `Epoxy for student.xlsx` | Epoxy resin price and its inputs (benzene, acetone, epichlorohydrin, other costs) | monthly, 2018-01 → 2026-10 | **target** (epoxy) + features |
| `benchmark-...-rm6-ne-iqi...xlsx` / `.pdf` | Should-cost teardown: Schneider RM6 vs 5 competitors (Ormazabal, SGC, ABB, Siemens, Eaton) | static, Nov 2026 | **context** (cost exposure per material) |
| `RMU_IQI_EU_Price_Benchmark_2017-2026_Synthetic.xlsx` | Protected with Microsoft Information Protection (Schneider tenant). **Cannot be read** outside Excel with a Schneider account | ? | **unavailable** |
| `Supplier_balancesheet/*.xlsx` | PitchBook financials for 9 suppliers | annual, ≤ 2025 | **context** (supplier risk) |

## 2. Official raw-material dataset (`rm_prices_monthly_wide.csv`)

Series prefixes: **S** = free public source, **D** = derived (`S01/S06`, `S02/S06` → EUR/t), **L** = licensed (empty on purpose).

In [ ]:
wide = pd.read_csv(CSV / "rm_prices_monthly_wide.csv", parse_dates=["month"], index_col="month")
catalog = pd.read_csv(CSV / "source_catalog.csv").set_index("series_id")
print(wide.shape, wide.index.min().date(), "→", wide.index.max().date())
catalog[["material", "region", "series_name", "unit", "access", "proxy_type"]]

In [ ]:
def coverage(df: pd.DataFrame) -> pd.DataFrame:
    """Per-column coverage: first/last observation, count, missing values inside the observed span."""
    rows = []
    for c in df.columns:
        s = df[c]
        nn = s.dropna()
        if nn.empty:
            rows.append(dict(series=c, n_obs=0, first=pd.NaT, last=pd.NaT, gaps_inside=np.nan, missing_total=len(s)))
            continue
        rows.append(dict(
            series=c, n_obs=len(nn), first=nn.index.min().date(), last=nn.index.max().date(),
            gaps_inside=int(s.loc[nn.index.min():nn.index.max()].isna().sum()),
            missing_total=int(s.isna().sum()),
        ))
    return pd.DataFrame(rows).set_index("series")


rm_cols = [c for c in wide.columns if c[0] in "SDL"]
coverage(wide[rm_cols])

Reading the table:

- **S/D series:** no internal gaps. The only missing value is the last month (Oct 2026), which hasn't been published yet.
  Only FX (`S06`) has Oct 2026.
- **L01–L08:** entirely empty. These are the licensed prices that match the BOM exactly (CR coil, 304L, HDG, PBT, PET, ETP,
  Al/Cu premiums). Section 3 shows that the steel workbook provides **L01–L03** (CR coil, 304, HDG). We have nothing for
  L04–L08 (plastics grades, Rotterdam premiums).

In [ ]:
def presence_map(df: pd.DataFrame, title: str, ax=None):
    """Observed (blue) vs missing (light grey) per series and month."""
    from matplotlib.colors import ListedColormap
    ax = ax or plt.gca()
    m = df.notna().T.astype(int).values
    ax.imshow(m, aspect="auto", interpolation="nearest", cmap=ListedColormap(["#e6e5e0", PALETTE[0]]))
    ax.set_yticks(range(df.shape[1]), df.columns)
    ticks = [i for i, d in enumerate(df.index) if d.month == 1]
    ax.set_xticks(ticks, [df.index[i].year for i in ticks])
    ax.grid(False)
    ax.set_title(title, loc="left")


fig, ax = plt.subplots(figsize=(11, 5))
presence_map(wide[rm_cols], "Raw-material dataset: observed (blue) vs missing (grey)", ax)
plt.tight_layout()

### 2.1 Levels

The series mix prices and indices with different base years, so the charts below are grouped by unit. Indices are
rebased to **2021 = 100** to be comparable.

In [ ]:
def small_multiples(df: pd.DataFrame, titles=None, ncols=3, height=2.3, color=PALETTE[0], sharex=True):
    cols = list(df.columns)
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.0 * ncols, height * nrows), sharex=sharex, squeeze=False)
    for ax, c in zip(axes.flat, cols):
        s = df[c].dropna()
        ax.plot(s.index, s.values, color=color)
        ax.set_title((titles or {}).get(c, c), loc="left")
        ax.xaxis.set_major_locator(mdates.YearLocator(2))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    for ax in axes.flat[len(cols):]:
        ax.set_visible(False)
    fig.tight_layout()
    return fig


price_titles = {c: f"{c} · {catalog.loc[c, 'material']} ({catalog.loc[c, 'unit']})" for c in
                ["S01", "S02", "S03", "S04", "S05", "S06", "D01", "D02"]}
small_multiples(wide[list(price_titles)], price_titles);

In [ ]:
idx_cols = ["S07", "S08", "S09", "S10", "S11", "S12", "S13"]
rebased = wide[idx_cols] / wide.loc["2021", idx_cols].mean() * 100
idx_titles = {c: f"{c} · {catalog.loc[c, 'material']} {catalog.loc[c, 'region']} (2021=100)" for c in idx_cols}
small_multiples(rebased, idx_titles);

The regimes listed in the README are clearly visible: the 2020 COVID dip, the 2021–22 commodity and energy surge, the
2022 nickel squeeze, and the 2025–26 copper rally to a record high. Any model has to cope with **regime changes**, and a
3-year-long surge-then-reversal makes up a large share of a 10-year sample.

## 3. Licensed steel prices (`Copia de Europe Steel INDEXES.xlsx`)

This workbook is Schneider's internal steel-index file. It contains **real transaction-price benchmarks in EUR/t**, which
the official dataset leaves empty (L01 CR coil, L02 stainless 304L, L03 HDG):

| Sheet | Content |
|---|---|
| `MEPS` | Monthly "low" values, HR / CR / HDG / EG coil for Germany, France, Italy, Spain, UK (GBP), Belgium (+ short Nordic/Sweden, wire rod, Fastmarkets Turkey, stainless 304/430) |
| `CRU Flat` | Weekly CRU flat prices + monthly average (Germany, Italy) or one spot value (France, Spain, UK) |
| `CRU Stainless` | Cold-rolled 2 mm 304 / 430 / 316: **base price + alloy surcharge** per country |
| `CRU Long` | Beams, merchant bar, rebar, wire rod since 1997 (not in our BOM) |
| `F▲`, `L▲`, `SS▲` | Month-on-month differences of the above (derived, not needed) |
| `Reference to Index Mapping`, `Q4-2017`, `UPDATE` | How Schneider plants index part prices to these steel indices (business context) |

The cells below parse the 3 sheets we need into tidy monthly tables. Dates are normalised to the first of the month
(CRU Stainless and CRU Long are stamped on the 15th).

In [ ]:
STEEL = RAW / "Copia de Europe Steel INDEXES.xlsx"


def to_month(idx) -> pd.DatetimeIndex:
    return pd.DatetimeIndex(pd.to_datetime(idx, errors="coerce")).to_period("M").to_timestamp()


def read_meps(path=STEEL) -> pd.DataFrame:
    raw = pd.read_excel(path, sheet_name="MEPS", header=None, engine="openpyxl")
    country, product, body = raw.iloc[0].ffill(), raw.iloc[4], raw.iloc[5:]
    cols = {}
    for c in range(1, 30):                       # Germany … Belgium blocks (the rest are short or off-scope)
        if pd.isna(product[c]):
            continue
        cols[f"MEPS {country[c]} {product[c]}"] = pd.to_numeric(body[c], errors="coerce").values
    df = pd.DataFrame(cols, index=to_month(body[0])).dropna(how="all")
    return df[df.index.notna()]


def read_cru_flat(path=STEEL) -> pd.DataFrame:
    raw = pd.read_excel(path, sheet_name="CRU Flat", header=None, engine="openpyxl")
    country, kind, product, market, body = raw.iloc[0].ffill(), raw.iloc[1], raw.iloc[2], raw.iloc[4], raw.iloc[6:]
    cols = {}
    for c in range(1, raw.shape[1]):
        if kind[c] in ("AVERAGE", "Spot price") and market[c] == "Domestic":
            cols[f"CRU {str(country[c]).title()} {product[c]}"] = pd.to_numeric(body[c], errors="coerce").values
    df = pd.DataFrame(cols, index=to_month(body[0])).dropna(how="all")
    return df[df.index.notna()]


def read_cru_stainless(path=STEEL) -> pd.DataFrame:
    """Transaction price = base + alloy surcharge, per country and grade."""
    raw = pd.read_excel(path, sheet_name="CRU Stainless", header=None, engine="openpyxl")
    country, grade, part, body = raw.iloc[0].ffill(), raw.iloc[1].ffill(), raw.iloc[4], raw.iloc[5:]
    idx = to_month(body[0])
    out = {}
    for c in range(1, raw.shape[1]):
        if part[c] != "BASE":
            continue
        base = pd.to_numeric(body[c], errors="coerce").values
        alloy = pd.to_numeric(body[c + 1], errors="coerce").values
        g = str(grade[c]).replace("Grade ", "")
        out[f"CRU {country[c]} {g} base"] = base
        out[f"CRU {country[c]} {g} alloy"] = alloy
        out[f"CRU {country[c]} {g} total"] = base + alloy
    df = pd.DataFrame(out, index=idx).dropna(how="all")
    return df[df.index.notna()]


meps = read_meps()
cru_flat = read_cru_flat()
cru_ss = read_cru_stainless()
print("MEPS", meps.shape, "| CRU flat", cru_flat.shape, "| CRU stainless", cru_ss.shape)

In [ ]:
steel_cov = pd.concat([
    coverage(meps).assign(sheet="MEPS"),
    coverage(cru_flat).assign(sheet="CRU Flat"),
    coverage(cru_ss[[c for c in cru_ss if c.endswith("total")]]).assign(sheet="CRU Stainless"),
])
steel_cov

Things to note:

- **MEPS flat** (all 6 countries) and **CRU flat** go to **Oct 2026**, one month *more* than the public PPIs. They also go
  back to 2012, which gives 4 more years of history than the public dataset (2016-11).
- **CRU stainless 304** ends **Aug 2026** for Germany and France. All other countries and grades stop in **Feb 2026**, so
  304 Germany/France is the usable stainless target.
- UK MEPS is in **GBP/t**, every other series in EUR/t.
- A few interior gaps exist (see `gaps_inside`). They will need imputation if those series are used.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.3), sharey=False)
for ax, prod in zip(axes, ["CR coil", "HDG coil", "HR coil"]):
    for i, ctry in enumerate(["Germany", "France", "Italy"]):
        s = meps[f"MEPS {ctry} {prod}"].dropna()
        ax.plot(s.index, s.values, color=PALETTE[i], label=ctry)
    ax.set_title(f"MEPS {prod} (EUR/t)", loc="left")
axes[0].legend()
fig.tight_layout()

fig, ax = plt.subplots(figsize=(13, 3.3))
for i, c in enumerate(["CRU Germany 304 total", "CRU France 304 total", "CRU Germany 304 base"]):
    s = cru_ss[c].dropna()
    ax.plot(s.index, s.values, color=PALETTE[i], label=c)
ax.set_title("CRU stainless 304 cold-rolled 2 mm (EUR/t): total = base + alloy surcharge", loc="left")
ax.legend()
fig.tight_layout()

### 3.1 Do the free proxies track the licensed prices?

The BOM maps carbon steel and stainless to the **EU steel PPI `S09`** and galvanised steel to the **German steel PPI
`S10`**. If we forecast the PPI instead of the actual price, we need to know how closely the two move together.
We compare monthly log-changes, at lag 0 and with the PPI shifted by ±1–3 months.

In [ ]:
def lag_corr(x: pd.Series, y: pd.Series, lags=range(-3, 4)) -> pd.Series:
    """corr(Δlog x_t, Δlog y_{t+lag}). Positive lag = x leads y."""
    dx, dy = np.log(x).diff(), np.log(y).diff()
    return pd.Series({lag: dx.corr(dy.shift(-lag)) for lag in lags})


pairs = {
    "MEPS DE CR coil vs S09": (meps["MEPS Germany CR coil"], wide["S09"]),
    "MEPS DE HDG coil vs S10": (meps["MEPS Germany HDG coil"], wide["S10"]),
    "CRU DE 304 total vs S09": (cru_ss["CRU Germany 304 total"], wide["S09"]),
    "CRU DE 304 alloy vs S03 nickel": (cru_ss["CRU Germany 304 alloy"], wide["S03"]),
    "MEPS DE CR vs CRU DE CR": (meps["MEPS Germany CR coil"], cru_flat["CRU Germany CR coil"]),
    "MEPS DE CR vs S04 iron ore": (meps["MEPS Germany CR coil"], wide["S04"]),
}
lc = pd.DataFrame({k: lag_corr(*v) for k, v in pairs.items()}).T
lc.columns = [f"lag {c:+d}" for c in lc.columns]
lc.round(2)

How to read it: *lag +k* = the first series leads the second by *k* months.

- **MEPS CR/HDG lead the PPIs by about 1 month** (peak at lag +1). The PPI is a lagging, smoothed echo of the market
  price, so it adds little as a *feature* for steel targets.
- **Nickel leads the stainless alloy surcharge by 1–2 months** (≈ 0.65). This is mechanical: the surcharge formula uses
  previous months' nickel prices. It is a usable leading feature for the 3-month horizon.
- **Iron ore leads MEPS CR by 1–3 months** (≈ 0.37–0.39).
- MEPS and CRU agree at lag 0 (0.93), so they are two measures of the same price. Either can be the target, and
  the other can fill gaps.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.3))
for ax, (lic, ppi, name) in zip(axes, [("MEPS Germany CR coil", "S09", "CR coil (MEPS DE) vs EU steel PPI S09"),
                                        ("MEPS Germany HDG coil", "S10", "HDG (MEPS DE) vs DE steel PPI S10")]):
    a = meps[lic].loc["2016-11":]
    b = wide[ppi]
    ax.plot(a.index, a / a.loc["2021"].mean() * 100, color=PALETTE[0], label=lic)
    ax.plot(b.index, b / b.loc["2021"].mean() * 100, color=PALETTE[1], label=ppi)
    ax.set_title(name + " — rebased 2021=100", loc="left")
    ax.legend()
fig.tight_layout()

The PPI is much **smoother** than the market price: the monthly volatility of MEPS DE CR is **≈ 2.7× that of S09**
(std of monthly log change 0.057 vs 0.021). The two also **drift apart in level**: today MEPS is ~24 % below its 2021
average, while S09 is ~5 % above (S09 also covers stainless and ferro-alloys). A model trained on the PPI would
underestimate the risk on the purchase price and get the level wrong. **The licensed MEPS/CRU series should be the
steel targets.**

In [ ]:
mapping = pd.read_excel(STEEL, sheet_name="Reference to Index Mapping", header=1, engine="openpyxl")
mapping.columns = [str(c).strip() for c in mapping.columns]
print(mapping.shape)
print(mapping.groupby(["Index", "Lead Time (Months)"]).size().rename("n_parts").sort_values(ascending=False).head(15))

The mapping sheet shows how Schneider uses these indices: each purchased part is linked to one steel index (e.g. `CR
France`, `HDG Germany`) with a **lead time** (lag between the index month and the price applied). This makes a
forecast of the index directly actionable for purchasing.

## 4. LME copper & aluminium workbook

A second source for copper and aluminium (Westmetall LME monthly averages). Compared with `S01/S02/D01/D02` it adds:

- **Oct 2026** as a completed month and **Nov 2026 month-to-date** (provisional, data to 11 Nov);
- the **3-month forward price** → the cash/3M spread (contango/backwardation), a market-implied expectation;
- **LME stocks** (tonnes), a supply-tightness signal.

In [ ]:
lme = pd.read_excel(RAW / "LME_Copper_Aluminium_EU_Monthly_2016_2026.xlsx", sheet_name="Monthly Prices",
                    header=4, engine="openpyxl")
lme["Month"] = to_month(lme["Month"])
lme = lme.set_index("Month")
print(lme["Period status"].value_counts().to_dict())
lme.tail(4)

In [ ]:
chk = pd.DataFrame({
    "Cu LME file (EUR/t)": lme["Cu cash EUR/t"], "D01": wide["D01"],
    "Al LME file (EUR/t)": lme["Al cash EUR/t"], "D02": wide["D02"],
}).dropna()
diff = pd.DataFrame({
    "Cu % diff": (chk.iloc[:, 0] / chk["D01"] - 1) * 100,
    "Al % diff": (chk.iloc[:, 2] / chk["D02"] - 1) * 100,
})
diff.describe().loc[["count", "mean", "50%", "min", "max"]]

In [ ]:
lme_done = lme[lme["Period status"] == "Completed month"]
fig, axes = plt.subplots(1, 2, figsize=(13, 3.3))
for ax, m, color in zip(axes, ["Cu", "Al"], PALETTE[:2]):
    ax.plot(lme_done.index, lme_done[f"{m} spread EUR/t"], color=color)
    ax.axhline(0, color=GREY, lw=0.8)
    ax.set_title(f"{m}: 3-month minus cash (EUR/t). <0 = backwardation", loc="left")
fig.tight_layout()

fig, axes = plt.subplots(1, 2, figsize=(13, 3.3))
for ax, m, color in zip(axes, ["Cu", "Al"], PALETTE[:2]):
    ax.plot(lme_done.index, lme_done[f"{m} avg stocks t"] / 1e3, color=color)
    ax.set_title(f"{m}: LME average stocks (kt)", loc="left")
fig.tight_layout()

The LME file matches `D01/D02` closely (see the diff table: differences in the 0.1–0.2 % range, as the README's quality
check says). We can safely **splice** it in to extend copper and aluminium to Oct 2026. The Nov MTD row is
provisional and should not be used as a training label.

## 5. Epoxy resin (`Epoxy for student.xlsx`)

Epoxy is not in the BOM CSV, but cast epoxy resin is the standard insulating material for RMU bushings and insulators.
The file gives the resin price and three chemical inputs, plus a "calculated" other-costs column.

In [ ]:
epoxy = pd.read_excel(RAW / "Epoxy for student.xlsx", engine="openpyxl")
epoxy = epoxy.rename(columns={epoxy.columns[0]: "date"}).drop(columns=epoxy.columns[1])
print("raw date stamps (day of month):", sorted(epoxy["date"].dt.day.unique()))
epoxy["month"] = to_month(epoxy["date"])
epoxy = epoxy.drop(columns="date").set_index("month")
epoxy.columns = ["benzene", "acetone", "epichlorohydrin", "other_costs_calc", "epoxy_resin"]
print("duplicate months:", epoxy.index.duplicated().sum())
coverage(epoxy)

In [ ]:
# Is the resin price an exact formula of its inputs? Fit a linear model on the full sample.
X = np.column_stack([np.ones(len(epoxy)), epoxy[["benzene", "acetone", "epichlorohydrin", "other_costs_calc"]].values])
y = epoxy["epoxy_resin"].values
beta, *_ = np.linalg.lstsq(X, y, rcond=None)
resid = y - X @ beta
print("coefficients (const, benzene, acetone, ECH, other):", np.round(beta, 3))
print(f"R² = {1 - resid.var() / y.var():.4f}, residual std = {resid.std():.1f} EUR/t")

small_multiples(epoxy, ncols=5, height=2.4);

## 6. Energy prices (features / leading indicators)

The energy dataset is designed as a set of **features**. Energy costs feed through to metal and plastic prices with a
delay. We check its coverage, then measure which energy series *lead* each target.

In [ ]:
energy = pd.read_csv(CSV / "energy_prices_monthly_wide.csv", parse_dates=["month"], index_col="month")
ecat = pd.read_csv(CSV / "energy_source_catalog.csv").set_index("series_id")
e_cols = [c for c in energy.columns if c[0] == "E"]
ecov = coverage(energy[e_cols]).join(ecat[["energy", "region", "unit"]] if "energy" in ecat else ecat.iloc[:, :3])
ecov

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
presence_map(energy[e_cols], "Energy dataset: observed (blue) vs missing (grey)", ax)
plt.tight_layout()

## 7. Bill of materials: what each product is made of

The BOM gives the weight of each forecast. There are **two versions** of it: `data/bom_materials.csv` and the
`BOM_Materials` sheet of the top-level `RM_Prices_XHEC_Challenge.xlsx`. They disagree.

In [ ]:
bom_csv = pd.read_csv(CSV / "bom_materials.csv")
bom_xlsx = pd.read_excel(RAW / "RM_Prices_XHEC_Challenge.xlsx", sheet_name="BOM_Materials", engine="openpyxl")
key = ["product_ref", "material_label_source"]
bom_cmp = (bom_csv[key + ["material_family", "share_pct", "mass_kg"]]
           .merge(bom_xlsx[key + ["share_pct", "mass_kg"]], on=key, how="outer", suffixes=("_csv", "_xlsx")))
bom_cmp = bom_cmp[bom_cmp["material_family"].notna()]
bom_cmp["mass_diff_kg"] = bom_cmp["mass_kg_csv"] - bom_cmp["mass_kg_xlsx"]
bom_cmp

To see which forecast matters most, we multiply the mass of each material by its latest price (EUR/t). This is a rough
exposure figure, not a cost: it ignores scrap, processing and the plastics, which only have an index. RM6's largest
slice (~45 %, unlabelled) is "most likely carbon steel" according to the README. We include it at **≈ [NDA] kg**, marked
as an assumption.

In [ ]:
last = lambda s: s.dropna().iloc[-1]
eur_per_t = {
    "Carbon steel": last(meps["MEPS Germany CR coil"]),
    "Carbon steel (to confirm)": last(meps["MEPS France CR coil"]),
    "Stainless steel": last(cru_ss["CRU Germany 304 total"]),
    "Galvanised steel": last(meps["MEPS Germany HDG coil"]),
    "Copper": last(lme_done["Cu cash EUR/t"]),
    "Aluminium": last(lme_done["Al cash EUR/t"]),
}
bom = bom_csv[bom_csv["plant_code"].notna()].copy()
bom.loc[bom["material_family"] == "Carbon steel (to confirm)", "mass_kg"] = 140.0   # README assumption
bom["eur_per_t"] = bom["material_family"].map(eur_per_t)
bom["exposure_eur"] = bom["mass_kg"] * bom["eur_per_t"] / 1000
exp = bom.dropna(subset=["exposure_eur"])[["product_ref", "material_family", "mass_kg", "eur_per_t", "exposure_eur"]]
exp["share_of_priced"] = exp["exposure_eur"] / exp.groupby("product_ref")["exposure_eur"].transform("sum")
exp.sort_values(["product_ref", "exposure_eur"], ascending=[True, False])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3))
fam_order = ["Carbon steel", "Carbon steel (to confirm)", "Stainless steel", "Galvanised steel", "Copper", "Aluminium"]
fam_color = dict(zip(fam_order, [PALETTE[0], PALETTE[0], PALETTE[1], PALETTE[2], PALETTE[3], PALETTE[4]]))
for ax, (prod, g) in zip(axes, exp.groupby("product_ref")):
    g = g.sort_values("exposure_eur")
    ax.barh(g["material_family"], g["exposure_eur"], color=[fam_color[f] for f in g["material_family"]], height=0.6)
    for y, v in enumerate(g["exposure_eur"]):
        ax.text(v, y, f"  €{v:,.0f}", va="center", fontsize=8, color="#52514e")
    ax.set_title(f"{prod}: mass × latest price (EUR / unit)", loc="left")
    ax.grid(axis="y", visible=False)
    ax.margins(x=0.2)
fig.tight_layout()

## 8. Should-cost benchmark & the protected RMU file

The benchmark workbook is a **teardown / should-cost** comparison of the Schneider RM6 against 5 competitors. It gives
cost breakdowns and part-level BOMs (material, process, mass). It is static (generated Nov 2026), so it is context for
the business recommendation, not training data.

In [ ]:
BENCH = next(RAW.glob("benchmark-*.xlsx"))
print(pd.ExcelFile(BENCH, engine="openpyxl").sheet_names)
bench = pd.read_excel(BENCH, sheet_name="Benchmark", header=None, engine="openpyxl")
bench.columns = ["field"] + [str(x).replace("\n", " ") for x in bench.iloc[4, 1:]]
bench.iloc[5:40].dropna(how="all", subset=bench.columns[1:]).set_index("field")

The benchmark is labelled *provisional*, and some columns are obviously incomplete. SGC's total should-cost is
€19 for a [NDA] kg unit (its notes say parts are "absent / assumed"). Use the per-material BOM sheets carefully.

In [ ]:
rmu = RAW / "RMU_IQI_EU_Price_Benchmark_2017-2026_Synthetic.xlsx"
try:
    pd.read_excel(rmu, engine="openpyxl")
except Exception as exc:
    head = rmu.read_bytes()[:8]
    print(f"Cannot read: {type(exc).__name__}: {exc}")
    print("File signature:", head.hex(), "→ OLE2 container (Microsoft Information Protection / RMS encrypted), not a zip-based xlsx")

## 9. Supplier financials (PitchBook)

Annual financials for 9 suppliers. These are useful for supplier-risk arguments in the business case. With at most
about 8 yearly points, they **cannot be monthly features** in a price model.

In [ ]:
def read_pitchbook_overview(path: Path) -> pd.DataFrame:
    raw = pd.read_excel(path, sheet_name="Overview", header=None, engine="openpyxl")
    hdr_row = next(i for i in range(raw.shape[0]) if raw.iloc[i].astype(str).str.match(r"^[A-Z][a-z]{2} \d{4}$").any())
    period_cols = [c for c in raw.columns if isinstance(raw.iat[hdr_row, c], str)
                   and pd.Series([raw.iat[hdr_row, c]]).str.match(r"^[A-Z][a-z]{2} \d{4}$").iloc[0]]
    body = raw.iloc[hdr_row + 1:, [2] + period_cols]
    body.columns = ["metric"] + [raw.iat[hdr_row, c] for c in period_cols]
    body = body.dropna(subset=["metric"])
    body["metric"] = body["metric"].astype(str).str.strip()
    body = body.set_index("metric")
    return body.apply(pd.to_numeric, errors="coerce").dropna(how="all")


rows = []
for f in sorted(SUP.glob("*.xlsx")):
    company = f.stem.split("PitchBook_-_")[1].split("_Company_Financials")[0].replace("_", " ")
    ov = read_pitchbook_overview(f)
    years = sorted(ov.columns, key=lambda c: pd.to_datetime(c, format="%b %Y"))
    ov = ov[years]                                   # oldest → latest
    pick = lambda m: ov.loc[m].dropna().iloc[-1] if m in ov.index and ov.loc[m].notna().any() else np.nan   # latest value
    rows.append(dict(company=company, n_years=len(years), oldest=years[0] if years else None,
                     latest=years[-1] if years else None,
                     revenue_keur=pick("Total Revenue"), ebitda_margin=pick("EBITDA Margin"),
                     debt_to_equity=pick("Total Debt") / pick("Total Equity")))
suppliers = pd.DataFrame(rows).set_index("company")
# PitchBook's own "Debt to Equity" row is 0 for every company, even when Total Debt > 0, so we recompute it.
suppliers

## 10. Targets and training data

### 10.1 Target definition

One monthly price series per BOM material, in **EUR/t**, choosing the series closest to what Schneider actually pays.

| Target | Series | Why |
|---|---|---|
| `carbon_steel_cr_de` | MEPS Germany CR coil | BOM "Carbon steel EU (CR coil)"; Germany = EU benchmark, Hungary plant |
| `carbon_steel_cr_fr` | MEPS France CR coil | France plant (RM6) |
| `galvanised_hdg_de` | MEPS Germany HDG coil | BOM "HDG Germany" |
| `stainless_304_de` | CRU Germany 304 base + alloy | BOM "Stainless steel EU 304L" |
| `copper_eur` | D01 extended with the LME file | LME in EUR/t |
| `aluminium_eur` | D02 extended with the LME file | LME in EUR/t |
| `plastics_eu_idx` | S11 (index) | No price is available for PBT / PET / ETP. Only an index can be forecast |
| `epoxy_resin` | Epoxy file | If epoxy is in scope |

**What we predict:** for an origin month *t* and horizon *h* ∈ {3, 6, 9}, the target is
`y(t, h) = log(P[t+h] / P[t])`, i.e. the **log return** over *h* months. The price forecast is `P[t] · exp(ŷ)`.
Returns are closer to stationary than levels, comparable across materials, and make the naive "no change" forecast
the natural baseline (ŷ = 0).

In [ ]:
def splice(primary: pd.Series, extension: pd.Series) -> pd.Series:
    """Use primary where available, fill later months from extension."""
    return primary.combine_first(extension)


targets = pd.DataFrame({
    "carbon_steel_cr_de": meps["MEPS Germany CR coil"],
    "carbon_steel_cr_fr": meps["MEPS France CR coil"],
    "galvanised_hdg_de": meps["MEPS Germany HDG coil"],
    "stainless_304_de": cru_ss["CRU Germany 304 total"],
    "copper_eur": splice(wide["D01"], lme_done["Cu cash EUR/t"]),
    "aluminium_eur": splice(wide["D02"], lme_done["Al cash EUR/t"]),
    "plastics_eu_idx": wide["S11"],
    "epoxy_resin": epoxy["epoxy_resin"],
}).sort_index()
targets = targets.loc["2008":]

tcov = coverage(targets)
tcov["months_behind_asof"] = [(ASOF.to_period("M") - pd.Period(d, "M")).n if pd.notna(d) else None for d in tcov["last"]]
tcov

In [ ]:
small_multiples(targets.loc["2012":], ncols=4, height=2.3);

### 10.2 Forecast origin

The targets end in different months (Aug, Sep or Oct 2026). We have to choose a **common forecast origin**. If the
origin is the latest month for which *every* target is known, the 3 / 6 / 9-month horizons land on the dates below.
Targets with more recent data can use it as extra information, but the publication lag has to be mirrored in training
(a feature known in month *t* must be available in month *t* in real time too).

The common origin is held back to **Aug 2026** only by CRU stainless 304. Every other target is known up to Sep or
Oct 2026.

In [ ]:
origin = pd.Timestamp(min(tcov["last"]))
latest = pd.Timestamp(max(tcov["last"]))
pd.DataFrame({
    "horizon": ["3 months", "6 months", "9 months"],
    "from common origin " + origin.strftime("%b %Y"): [(origin + pd.DateOffset(months=h)).strftime("%b %Y") for h in (3, 6, 9)],
    "from latest data " + latest.strftime("%b %Y"): [(latest + pd.DateOffset(months=h)).strftime("%b %Y") for h in (3, 6, 9)],
}).set_index("horizon")

### 10.3 How hard is each horizon? Naive baseline

For each target and horizon: the number of training pairs (*t*, *t+h*) available, the volatility of the *h*-month log
return, and the error of the **random-walk** forecast (tomorrow = today). Any model must beat this baseline. Counts are
shown for two windows: the full history of each series, and the **common window from 2016-11**, where all the public
features (energy, PPIs) exist.

In [ ]:
H = (3, 6, 9)


def horizon_stats(df: pd.DataFrame, start=None) -> pd.DataFrame:
    rows = []
    for c in df.columns:
        s = df[c].loc[start:].dropna() if start else df[c].dropna()
        for h in H:
            r = np.log(s.shift(-h) / s).dropna()
            rows.append(dict(target=c, h=h, n_pairs=len(r), ret_std=r.std(),
                             naive_mape=(np.exp(r) - 1).abs().mean() * 100,
                             share_up=(r > 0).mean()))
    return pd.DataFrame(rows).set_index(["target", "h"])


hs_full = horizon_stats(targets)
hs_common = horizon_stats(targets, start="2016-11")
tab = pd.concat({"full history": hs_full, "from 2016-11": hs_common}, axis=1)
tab[[("full history", "n_pairs"), ("from 2016-11", "n_pairs"),
     ("from 2016-11", "ret_std"), ("from 2016-11", "naive_mape"), ("from 2016-11", "share_up")]].unstack("h")

In [ ]:
m = hs_common["naive_mape"].unstack("h")
m = m.loc[m[9].sort_values().index]
fig, ax = plt.subplots(figsize=(10, 3.6))
y = np.arange(len(m))
for i, h in enumerate(H):
    ax.barh(y + (i - 1) * 0.26, m[h], height=0.24, color=PALETTE[i], label=f"h = {h} months")
ax.set_yticks(y, m.index)
ax.set_xlabel("Random-walk MAPE (%) — from 2016-11")
ax.set_title("Baseline to beat: error of the 'no change' forecast", loc="left")
ax.grid(axis="y", visible=False)
ax.legend(loc="lower right")
fig.tight_layout()

### 0 Which features lead the targets?

Correlation between the monthly log change of each candidate feature in month *t* and the target's log change
**over the next 3 months** (*t* → *t+3*). This is a rough screening: any value of about |0.2| or more on ~115 points
deserves a closer look. (|r| > 0.18 is significant at 5 % only under independence; the overlapping 3-month windows
make the true threshold higher.)

In [ ]:
feat = pd.concat([
    wide[["S01", "S02", "S03", "S04", "S05", "S06", "S07", "S08", "S09", "S10", "S11", "S12", "S13"]],
    energy[["E01", "E05", "E09", "E10", "E11", "E13", "E22", "ED02"]],
], axis=1)
feat["Cu_spread"] = lme_done["Cu spread EUR/t"] / lme_done["Cu cash EUR/t"]
feat["Al_spread"] = lme_done["Al spread EUR/t"] / lme_done["Al cash EUR/t"]
feat["Cu_stocks"] = lme_done["Cu avg stocks t"]
feat["Al_stocks"] = lme_done["Al avg stocks t"]
feat = feat.loc["2016-11":]

d_feat = feat.apply(lambda s: s.diff() if s.name.endswith("_spread") else np.log(s).diff())
fwd3 = np.log(targets.shift(-3) / targets).loc["2016-11":]
lead = pd.DataFrame({t: d_feat.corrwith(fwd3[t]) for t in targets.columns})
lead.loc["own Δ1m (momentum)"] = [np.log(targets[t]).diff().loc["2016-11":].corr(fwd3[t]) for t in targets.columns]
lead.round(2)

How to read it:

- **Steel (CR, HDG, 304)** has the most predictable structure. Last month's broad metals moves (Cu, Al, iron ore:
  ≈ 0.42–0.46) **and the steel price's own momentum** (≈ 0.26) carry into the next 3 months. Steel prices trend,
  so momentum / autoregressive features should be useful here.
- **Plastics (S11) and epoxy** are led by upstream energy and the other PPIs (Brent `E01` ≈ 0.52, IMF energy index
  `E22` ≈ 0.52, aluminium PPI `S08` ≈ 0.58). This matches the README's "Brent leads plastics by 1 month" finding.
  The plastics index also trends strongly (own momentum 0.44). **Epoxy does the opposite: its own momentum is −0.24,
  so it mean-reverts** (a month-on-month jump tends to partly reverse).
- **Copper and aluminium** show almost **no** leading signal (all |r| ≤ 0.2). These are liquid exchange-traded markets,
  so the random walk will be hard to beat. Expect most of the value there to come from ranges (prediction intervals),
  not point forecasts.
- **Spreads and LME stocks** add little on this screen. They may still help in non-linear models or regime detection.

## 11. Summary

### What we have

- **Targets in EUR/t for every priceable BOM metal.** CR coil, HDG coil and stainless 304 come from the licensed
  MEPS / CRU sheets of the steel workbook. Copper and aluminium come from the LME (official D01/D02 extended to Oct 2026
  by the LME workbook, which matches them within ±1 %). We also have epoxy resin in EUR/t.
- **Features:** 13 public metal / PPI / FX series, 30 energy series, LME 3M spreads and stocks, epoxy feedstocks
  (benzene, acetone, epichlorohydrin), and other steel products and countries (HR coil, Italy/Spain/Belgium, CRU long).
- **Clean data:** no interior gaps in the official datasets. The only blanks are the not-yet-published last months.
- **Weights:** BOM masses for both products. By value (mass × price), **RM6 is driven by copper (~63 %)**, then
  stainless and carbon steel. **RM AirSeT is driven by stainless 304 (~50 %) and CR carbon steel (~48 %)**.

### What is missing or unreliable

| Gap | Impact | Workaround |
|---|---|---|
| L04–L06: PBT, PET, ETP prices | Plastics can only be forecast as an **index** (S11) | Forecast % change of S11 and apply it to a reference price if one is available |
| L07–L08: Rotterdam Al/Cu premiums | Physical premium over LME not modelled | Ignore or treat as a constant |
| CRU stainless ends Aug 2026 (DE/FR), Feb 2026 (others) | Stainless is 1–2 months staler than the other targets | Nowcast Sep–Oct from nickel (S03 leads the alloy surcharge by 1–2 months) |
| Publication lags (PPIs/IMF to Sep, Eurostat oil imports to Jul) | Leakage risk if features are aligned naively | Lag each feature by its real publication delay when building training rows |
| Two different BOMs (CSV vs top-level xlsx, up to ±[NDA] kg) | Weights uncertain | Confirm with the challenge organisers. The CSV is the documented one |
| RM6 largest slice (~45 %) unlabelled | Carbon steel mass in RM6 is an assumption (~[NDA] kg) | Flag in the business case |
| `RMU_IQI_EU_Price_Benchmark…Synthetic.xlsx` is RMS-encrypted | Unknown content (probably product selling prices) | Ask for an unprotected copy |
| Supplier financials | Annual, 1–8 years, some fields empty | Context only (supplier risk), not a model input |
| Should-cost benchmark | Provisional, some columns incomplete | Context only |

### Training data

- **Rows:** one per month (*t*). **Label:** `log(P[t+h] / P[t])` for *h* = 3, 6, 9, one model (or one output) per
  horizon and per target.
- **Sample size is small:** on the common window from 2016-11, **~117 / 114 / 111 pairs** for *h* = 3 / 6 / 9. Steel
  goes back to 2012 (~165 pairs) and stainless to 2008 (~220) if we accept fewer features for those years. The pairs
  overlap (consecutive 9-month windows share 8 months), so the *effective* sample is much smaller. Use
  **expanding-window time-series CV**, never random splits, with a gap of *h* months between train and test.
- **Baseline to beat:** random walk, MAPE ≈ **7–10 % at 3 months, 10–14 % at 6 months, 12–19 % at 9 months** for
  the metals, lower for plastics (3.5 → 8.4 %) and epoxy (4.7 → 9.9 %).
- **Regime changes** (COVID, 2021–22 surge, energy shock, 2025–26 copper rally) dominate the sample. Plan for regime
  features or robust losses, and report prediction intervals rather than point forecasts only.

### Suggested next steps

1. Build a `features.py` that assembles one aligned monthly table (targets + features) with publication-lag shifts.
2. Baselines: random walk, seasonal naive, AR on log returns; then gradient boosting / ridge on lagged features.
3. Evaluate per target × horizon with rolling-origin CV, against the random walk (MAPE and directional accuracy).